# HAM10000 Pilot — MedGemma-4B-it (Zero-Shot Skin Lesion Classification)

**Model:** google/medgemma-4b-it (zero-shot, no fine-tuning, no few-shot examples)
**Sample:** same 105 images (15 per class) as the Qwen2.5-VL run — identical seed, so this is a fair head-to-head comparison.
**Requires:** Hugging Face access to google/medgemma-4b-it (accept the license) and an HF_TOKEN.

Run all cells top to bottom on a Colab or Kaggle GPU runtime (T4 or better).

In [2]:
!pip install -q transformers accelerate pillow pandas scikit-learn torch

## Step 0 — Hugging Face login

Make sure you've accepted the license at huggingface.co/google/medgemma-4b-it first.

In [6]:
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


In [13]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(hf_token)

## Step 1 — Point this at your HAM10000 data

Same as before:
- **Kaggle:** Add Data → "Skin Cancer MNIST: HAM10000" → check the actual mount path in the sidebar
  (it may be `/kaggle/input/skin-cancer-mnist-ham10000/` or nested under `/kaggle/input/datasets/kmader/...`).
- **Colab:** download via the Kaggle API as before.

In [8]:
import pandas as pd
from pathlib import Path

# EDIT THIS to match your actual mount path
DATA_DIR = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")

metadata = pd.read_csv(DATA_DIR / "HAM10000_metadata.csv")
print(metadata["dx"].value_counts())

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


## Step 2 — Build the image path lookup and label names (same as before)

In [9]:
import os

img_folders = [DATA_DIR / "HAM10000_images_part_1", DATA_DIR / "HAM10000_images_part_2"]
image_path_lookup = {}
for folder in img_folders:
    if folder.exists():
        for f in os.listdir(folder):
            if f.endswith(".jpg"):
                image_path_lookup[f.replace(".jpg", "")] = str(folder / f)

metadata["image_path"] = metadata["image_id"].map(image_path_lookup)
metadata = metadata.dropna(subset=["image_path"]).reset_index(drop=True)
print(f"Images found on disk: {len(metadata)}")

LABEL_NAMES = {
    "akiec": "Actinic keratoses / intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions",
}

Images found on disk: 10015


## Step 3 — Same stratified sample (SAME seed = same 105 images as the Qwen run)

In [10]:
SEED = 42
N_PER_CLASS = 15

sampled = (
    metadata.groupby("dx", group_keys=False)
    .apply(lambda g: g.sample(n=min(N_PER_CLASS, len(g)), random_state=SEED))
    .reset_index(drop=True)
)
print(f"Total sampled: {len(sampled)}")
sampled.to_csv("ham10000_pilot_sample_medgemma.csv", index=False)
sampled[["image_id", "dx"]].head(10)

Total sampled: 105


/tmp/ipykernel_58/593048432.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(N_PER_CLASS, len(g)), random_state=SEED))


,image_id,dx
0,ISIC_0024646,akiec
1,ISIC_0027753,akiec
2,ISIC_0032404,akiec
3,ISIC_0029041,akiec
4,ISIC_0030491,akiec
5,ISIC_0026626,akiec
6,ISIC_0026702,akiec
7,ISIC_0030730,akiec
8,ISIC_0032014,akiec
9,ISIC_0027303,akiec


## Step 4 — Load MedGemma-4B-it

MedGemma uses the Gemma 3 multimodal architecture, so it loads via `AutoModelForImageTextToText`
(different class than Qwen2.5-VL) and uses `apply_chat_template` directly for both text and image.

In [15]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-4b-it"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

## Step 5 — Zero-shot prompt and inference loop

Same prompt wording as the Qwen run, so any difference in results reflects the model, not the prompt.

In [17]:
from PIL import Image

CLASS_LIST_TEXT = "\n".join(f"- {code_}: {name}" for code_, name in LABEL_NAMES.items())

def build_prompt():
    return (
        "You are a dermatology assistant. Look at this dermatoscopic image and classify it "
        "into exactly ONE of the following categories. Respond with ONLY the category code "
        "(e.g. 'mel'), nothing else.\n\n"
        f"Categories:\n{CLASS_LIST_TEXT}"
    )

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": build_prompt()},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        generation = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        generation = generation[0][input_len:]
    return processor.decode(generation, skip_special_tokens=True).strip()

def parse_label(raw_output):
    lowered = raw_output.lower()
    for code_ in LABEL_NAMES:
        if code_ in lowered:
            return code_
    return "unparsed"


In [18]:
results = []
for i, row in sampled.iterrows():
    raw = predict(row["image_path"])
    pred = parse_label(raw)
    results.append({
        "image_id": row["image_id"],
        "true_label": row["dx"],
        "predicted_label": pred,
        "raw_output": raw,
    })
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(sampled)} done")

results_df = pd.DataFrame(results)
results_df.to_csv("ham10000_pilot_results_medgemma.csv", index=False)
results_df.head()

10/105 done
20/105 done
30/105 done
40/105 done
50/105 done
60/105 done
70/105 done
80/105 done
90/105 done
100/105 done


,image_id,true_label,predicted_label,raw_output
0,ISIC_0024646,akiec,mel,mel
1,ISIC_0027753,akiec,mel,mel
2,ISIC_0032404,akiec,nv,nv
3,ISIC_0029041,akiec,mel,mel
4,ISIC_0030491,akiec,mel,mel


## Step 6 — Score the results

In [19]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

valid = results_df[results_df["predicted_label"] != "unparsed"]
print(f"Parsed cleanly: {len(valid)}/{len(results_df)}")

acc = accuracy_score(valid["true_label"], valid["predicted_label"])
print(f"Overall accuracy: {acc:.3f}\n")

print("Per-class report:")
print(classification_report(valid["true_label"], valid["predicted_label"], zero_division=0))

labels = list(LABEL_NAMES.keys())
cm = confusion_matrix(valid["true_label"], valid["predicted_label"], labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_df.to_csv("ham10000_pilot_confusion_matrix_medgemma.csv")
cm_df

Parsed cleanly: 98/105
Overall accuracy: 0.173

Per-class report:
              precision    recall  f1-score   support

       akiec       0.00      0.00      0.00        15
         bcc       0.06      0.07      0.06        15
         bkl       0.33      0.07      0.12        14
          df       0.00      0.00      0.00        12
         mel       0.22      0.80      0.34        15
          nv       0.13      0.21      0.16        14
        vasc       0.00      0.00      0.00        13

    accuracy                           0.17        98
   macro avg       0.11      0.16      0.10        98
weighted avg       0.11      0.17      0.10        98



,pred_akiec,pred_bcc,pred_bkl,pred_df,pred_mel,pred_nv,pred_vasc
true_akiec,0,1,0,0,11,3,0
true_bcc,0,1,0,0,7,7,0
true_bkl,0,0,1,0,8,5,0
true_df,0,5,1,0,4,2,0
true_mel,0,0,0,0,12,3,0
true_nv,0,0,0,0,10,3,1
true_vasc,0,9,1,0,3,0,0


## Step 7 — Misclassified cases

In [20]:
wrong = results_df[results_df["predicted_label"] != results_df["true_label"]]
print(f"{len(wrong)} misclassified out of {len(results_df)}")
wrong[["image_id", "true_label", "predicted_label", "raw_output"]]

88 misclassified out of 105


,image_id,true_label,predicted_label,raw_output
0,ISIC_0024646,akiec,mel,mel
1,ISIC_0027753,akiec,mel,mel
2,ISIC_0032404,akiec,nv,nv
3,ISIC_0029041,akiec,mel,mel
4,ISIC_0030491,akiec,mel,mel
...,...,...,...,...
100,ISIC_0032538,vasc,bcc,bcc
101,ISIC_0029608,vasc,unparsed,bcl
102,ISIC_0024475,vasc,bcc,bcc
103,ISIC_0026490,vasc,bcc,bcc
